# Module 3: Data Pipeline

Training an LLM requires massive amounts of text data. In this notebook, we'll explore how nanochat handles data loading and preprocessing.

## What You'll Learn

- **FinWeb dataset** - High-quality web text for pretraining
- **Data sharding** - Splitting data for distributed training
- **Tokenizing data loader** - On-the-fly tokenization
- **Sequence packing** - Efficient use of context window
- **Distributed data loading** - Multi-GPU coordination

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path

# Check if we're in the nanochat directory
if not os.path.exists('nanochat'):
    %cd nanochat

print(f"Working directory: {os.getcwd()}")

## 3.1 The FinWeb Dataset

Nanochat pretrains on **FinWeb** (FineWeb-Edu), a high-quality subset of web text.

Key characteristics:
- Filtered for educational content
- Deduplicated to reduce repetition
- ~15 trillion tokens in full dataset
- Nanochat uses ~50B tokens (subset)

In [ ]:
# Let's see how nanochat downloads data
# From nanochat/dataset.py:

FINWEB_DATASET_NAME = "HuggingFaceFW/fineweb-edu"
SHARD_SIZE_CHARS = 100_000_000  # 100M characters per shard

print("FinWeb-Edu Dataset:")
print(f"  - HuggingFace path: {FINWEB_DATASET_NAME}")
print(f"  - Shard size: {SHARD_SIZE_CHARS:,} characters (~100MB)")
print(f"  - Format: JSONL with 'text' field")
print("\nData flow:")
print("  1. Download from HuggingFace (streaming)")
print("  2. Split into shards of 100M chars each")
print("  3. Save as text files")
print("  4. Tokenize on-the-fly during training")

In [ ]:
# Example: Download a small sample
from datasets import load_dataset

# Load a tiny sample for demonstration
dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)

# Get first 3 examples
samples = list(dataset.take(3))

print("Sample documents from FinWeb-Edu:")
print("="*60)
for i, sample in enumerate(samples):
    text = sample['text']
    print(f"\n--- Document {i+1} ({len(text):,} chars) ---")
    print(text[:500] + "..." if len(text) > 500 else text)
    print()

## 3.2 Data Sharding

For distributed training, data is split into shards:

```
data/
├── train_000.txt  (100M chars)
├── train_001.txt  (100M chars)
├── ...
├── train_019.txt  (100M chars)  # 2B chars total for $100 tier
└── val_000.txt    (validation)
```

In [ ]:
def simulate_sharding(total_chars, shard_size, num_gpus):
    """Simulate how data is distributed across GPUs."""
    num_shards = total_chars // shard_size
    shards_per_gpu = num_shards // num_gpus
    
    print(f"Total data: {total_chars / 1e9:.1f}B characters")
    print(f"Shard size: {shard_size / 1e6:.0f}M characters")
    print(f"Number of shards: {num_shards}")
    print(f"Number of GPUs: {num_gpus}")
    print(f"Shards per GPU: {shards_per_gpu}")
    print(f"Characters per GPU: {shards_per_gpu * shard_size / 1e9:.2f}B")
    
    return num_shards, shards_per_gpu

print("$100 tier (2B chars, 8 GPUs):")
simulate_sharding(2_000_000_000, 100_000_000, 8)

print("\n$1000 tier (4B chars, 8 GPUs):")
simulate_sharding(4_000_000_000, 100_000_000, 8)

## 3.3 Tokenizing Data Loader

Nanochat tokenizes data **on-the-fly** during training, which:
- Saves disk space (no pre-tokenized data)
- Allows flexible tokenizer changes
- Uses CPU while GPU trains

In [ ]:
import tiktoken

class TokenizingDataLoader:
    """Simplified version of nanochat's data loader."""
    
    def __init__(self, text_files, tokenizer, batch_size, seq_len, device):
        self.text_files = text_files
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.device = device
        
        # Buffer for tokens
        self.token_buffer = []
        self.file_idx = 0
        self.char_offset = 0
    
    def _refill_buffer(self, min_tokens):
        """Read more text and tokenize."""
        while len(self.token_buffer) < min_tokens:
            if self.file_idx >= len(self.text_files):
                self.file_idx = 0  # Wrap around
            
            # Read chunk of text
            with open(self.text_files[self.file_idx], 'r') as f:
                f.seek(self.char_offset)
                text = f.read(1_000_000)  # 1M chars at a time
                
            if len(text) == 0:
                # Move to next file
                self.file_idx += 1
                self.char_offset = 0
                continue
            
            self.char_offset += len(text)
            
            # Tokenize
            tokens = self.tokenizer.encode(text)
            self.token_buffer.extend(tokens)
    
    def get_batch(self):
        """Get a batch of (input, target) pairs."""
        tokens_needed = self.batch_size * (self.seq_len + 1)
        self._refill_buffer(tokens_needed)
        
        # Take tokens from buffer
        tokens = self.token_buffer[:tokens_needed]
        self.token_buffer = self.token_buffer[tokens_needed:]
        
        # Reshape into batches
        tokens = torch.tensor(tokens, dtype=torch.long)
        tokens = tokens.view(self.batch_size, self.seq_len + 1)
        
        x = tokens[:, :-1].to(self.device)  # Input
        y = tokens[:, 1:].to(self.device)   # Target (shifted by 1)
        
        return x, y

print("TokenizingDataLoader workflow:")
print("  1. Read text chunk from file")
print("  2. Tokenize with BPE tokenizer")
print("  3. Fill token buffer")
print("  4. Extract batch of (input, target) pairs")
print("  5. Move to GPU for training")

## 3.4 Sequence Packing

Documents are concatenated with `<|bos|>` separators and packed into fixed-length sequences:

In [ ]:
# Demonstrate sequence packing
enc = tiktoken.get_encoding("gpt2")
BOS = "<|endoftext|>"  # GPT-2's equivalent of <|bos|>
bos_token = enc.encode(BOS, allowed_special={BOS})[0]

documents = [
    "The quick brown fox jumps over the lazy dog.",
    "Hello world! This is a test.",
    "Machine learning is fascinating."
]

# Tokenize with BOS
all_tokens = []
for doc in documents:
    all_tokens.append(bos_token)
    all_tokens.extend(enc.encode(doc))

print(f"Total tokens: {len(all_tokens)}")
print(f"Token sequence: {all_tokens}")

# Pack into sequences of length 16
seq_len = 16
num_seqs = len(all_tokens) // seq_len

print(f"\nPacked into {num_seqs} sequences of length {seq_len}:")
for i in range(num_seqs):
    seq = all_tokens[i*seq_len:(i+1)*seq_len]
    decoded = enc.decode(seq)
    print(f"  Seq {i}: {seq}")
    print(f"         '{decoded}'")

## 3.5 Distributed Data Loading

For multi-GPU training, each GPU gets different data:

```
GPU 0: shards [0, 8, 16, ...]
GPU 1: shards [1, 9, 17, ...]
GPU 2: shards [2, 10, 18, ...]
...
```

In [ ]:
def distribute_shards(num_shards, world_size, rank):
    """Assign shards to each GPU rank."""
    all_shards = list(range(num_shards))
    my_shards = all_shards[rank::world_size]  # Stride by world_size
    return my_shards

# Example with 20 shards and 8 GPUs
num_shards = 20
world_size = 8

print(f"Distributing {num_shards} shards across {world_size} GPUs:")
print("="*50)
for rank in range(world_size):
    shards = distribute_shards(num_shards, world_size, rank)
    print(f"  GPU {rank}: shards {shards}")

print("\nThis ensures:")
print("  - Each GPU processes different data")
print("  - Load is balanced across GPUs")
print("  - No data duplication within an epoch")

## 3.6 Batch Size and Gradient Accumulation

The effective batch size is:

```
total_batch_size = device_batch_size × seq_len × world_size × grad_accum_steps
```

In [ ]:
def compute_grad_accum(target_batch_size, device_batch_size, seq_len, world_size):
    """Compute gradient accumulation steps needed."""
    tokens_per_step = device_batch_size * seq_len * world_size
    grad_accum = target_batch_size // tokens_per_step
    
    print(f"Configuration:")
    print(f"  - Target batch size: {target_batch_size:,} tokens")
    print(f"  - Device batch size: {device_batch_size}")
    print(f"  - Sequence length: {seq_len}")
    print(f"  - World size (GPUs): {world_size}")
    print(f"\nCalculation:")
    print(f"  - Tokens per micro-batch: {device_batch_size} × {seq_len} = {device_batch_size * seq_len:,}")
    print(f"  - Tokens per step (all GPUs): {tokens_per_step:,}")
    print(f"  - Gradient accumulation steps: {grad_accum}")
    print(f"  - Actual batch size: {tokens_per_step * grad_accum:,} tokens")
    
    return grad_accum

# Nanochat's default settings
compute_grad_accum(
    target_batch_size=524288,  # 512K tokens
    device_batch_size=32,
    seq_len=2048,
    world_size=8
)

## 3.7 Data for Fine-tuning (Chat Format)

For SFT/chat training, data is in conversation format:

In [ ]:
import json

# Example conversation format
conversation = {
    "messages": [
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."},
        {"role": "user", "content": "What's its population?"},
        {"role": "assistant", "content": "Paris has about 2.2 million people."}
    ]
}

print("Chat format (JSONL):")
print(json.dumps(conversation, indent=2))

print("\nTokenized format:")
print("<|bos|><|user_start|>What is the capital of France?<|user_end|>")
print("<|assistant_start|>The capital of France is Paris.<|assistant_end|>")
print("<|user_start|>What's its population?<|user_end|>")
print("<|assistant_start|>Paris has about 2.2 million people.<|assistant_end|>")

## Summary

In this notebook, we learned:

1. ✅ **FinWeb dataset** - High-quality web text source
2. ✅ **Data sharding** - Splitting for parallel processing
3. ✅ **On-the-fly tokenization** - Flexible, memory-efficient
4. ✅ **Sequence packing** - Efficient context window usage
5. ✅ **Distributed loading** - Multi-GPU data distribution
6. ✅ **Batch size calculation** - Gradient accumulation

## Next Steps

Continue to **[Module 4: Pretraining](04_pretraining.ipynb)** to learn:
- Training loop implementation
- Muon optimizer
- Learning rate scheduling
- Distributed Data Parallel (DDP)

---

**Estimated time for this notebook: 30-40 minutes**